# RFM ile Müşteri Segmentasyonu (Customer Segmentation with RFM)

FLO müşteri verisi üzerinde RFM analizi ile segmentasyon çalışması.

## İş Problemi (Business Problem)

FLO müşterilerini segmentlere ayırıp bu segmentlere göre pazarlama stratejileri belirlemek istiyor.
Buna yönelik olarak müşterilerin davranışları tanımlanacak ve bu davranış öbeklenmelerine göre gruplar oluşturulacak.

## Veri Seti Hikayesi

Veri seti, son alışverişlerini 2020 - 2021 yıllarında OmniChannel (hem online hem offline alışveriş yapan) olarak yapan müşterilerin geçmiş alışveriş davranışlarından elde edilen bilgilerden oluşmaktadır.

**Değişkenler**

- **master_id:** Eşsiz müşteri numarası
- **order_channel:** Alışveriş yapılan platforma ait hangi kanalın kullanıldığı (Android, ios, Desktop, Mobile, Offline)
- **last_order_channel:** En son alışverişin yapıldığı kanal
- **first_order_date:** Müşterinin yaptığı ilk alışveriş tarihi
- **last_order_date:** Müşterinin yaptığı son alışveriş tarihi
- **last_order_date_online:** Müşterinin online platformda yaptığı son alışveriş tarihi
- **last_order_date_offline:** Müşterinin offline platformda yaptığı son alışveriş tarihi
- **order_num_total_ever_online:** Müşterinin online platformda yaptığı toplam alışveriş sayısı
- **order_num_total_ever_offline:** Müşterinin offline'da yaptığı toplam alışveriş sayısı
- **customer_value_total_ever_offline:** Müşterinin offline alışverişlerinde ödediği toplam ücret
- **customer_value_total_ever_online:** Müşterinin online alışverişlerinde ödediği toplam ücret
- **interested_in_categories_12:** Müşterinin son 12 ayda alışveriş yaptığı kategorilerin listesi

## Görevler

1. **GÖREV 1:** Veriyi Anlama (Data Understanding) ve Hazırlama
2. **GÖREV 2:** RFM Metriklerinin Hesaplanması
3. **GÖREV 3:** RF ve RFM Skorlarının Hesaplanması
4. **GÖREV 4:** RF Skorlarının Segment Olarak Tanımlanması
5. **GÖREV 5:** Aksiyon zamanı!
6. **GÖREV 6:** Tüm süreci fonksiyonlaştırınız.

In [24]:
import datetime as dt
import pandas as pd
import numpy as np


pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.3f' % x)

---
## GÖREV 1: Veriyi Hazırlama ve Anlama (Data Understanding)

### 1. Veriyi Okuma

`flo_data_20k.csv` verisini okuyunuz.

In [25]:
df = pd.read_csv("datasets/flo_data_20k.csv")
df_copy = df.copy()


### 2. Veri Seti İncelemesi

Veri setinde aşağıdaki incelemeleri yapınız:

- a. İlk 10 gözlem
- b. Değişken isimleri
- c. Boyut
- d. Betimsel istatistik
- e. Boş değer
- f. Değişken tipleri

In [26]:
df_copy.head(10)

,master_id,order_channel,last_order_channel,first_order_date,last_order_date,last_order_date_online,last_order_date_offline,order_num_total_ever_online,order_num_total_ever_offline,customer_value_total_ever_offline,customer_value_total_ever_online,interested_in_categories_12
0,cc294636-19f0-11eb-8d74-000d3a38a36f,Android App,Offline,2020-10-30,2021-02-26,2021-02-21,2021-02-26,4.000,1.000,139.990,799.380,[KADIN]
1,f431bd5a-ab7b-11e9-a2fc-000d3a38a36f,Android App,Mobile,2017-02-08,2021-02-16,2021-02-16,2020-01-10,19.000,2.000,159.970,1853.580,"[ERKEK, COCUK, KADIN, AKTIFSPOR]"
2,69b69676-1a40-11ea-941b-000d3a38a36f,Android App,Android App,2019-11-27,2020-11-27,2020-11-27,2019-12-01,3.000,2.000,189.970,395.350,"[ERKEK, KADIN]"
3,1854e56c-491f-11eb-806e-000d3a38a36f,Android App,Android App,2021-01-06,2021-01-17,2021-01-17,2021-01-06,1.000,1.000,39.990,81.980,"[AKTIFCOCUK, COCUK]"
4,d6ea1074-f1f5-11e9-9346-000d3a38a36f,Desktop,Desktop,2019-08-03,2021-03-07,2021-03-07,2019-08-03,1.000,1.000,49.990,159.990,[AKTIFSPOR]
5,e585280e-aae1-11e9-a2fc-000d3a38a36f,Desktop,Offline,2018-11-18,2021-03-13,2018-11-18,2021-03-13,1.000,2.000,150.870,49.990,[KADIN]
6,c445e4ee-6242-11ea-9d1a-000d3a38a36f,Android App,Android App,2020-03-04,2020-10-18,2020-10-18,2020-03-04,3.000,1.000,59.990,315.940,[AKTIFSPOR]
7,3f1b4dc8-8a7d-11ea-8ec0-000d3a38a36f,Mobile,Offline,2020-05-15,2020-08-12,2020-05-15,2020-08-12,1.000,1.000,49.990,113.640,[COCUK]
8,cfbda69e-5b4f-11ea-aca7-000d3a38a36f,Android App,Android App,2020-01-23,2021-03-07,2021-03-07,2020-01-25,3.000,2.000,120.480,934.210,"[ERKEK, COCUK, KADIN]"
9,1143f032-440d-11ea-8b43-000d3a38a36f,Mobile,Mobile,2019-07-30,2020-10-04,2020-10-04,2019-07-30,1.000,1.000,69.980,95.980,"[KADIN, AKTIFSPOR]"


In [27]:
df_copy.columns

Index(['master_id', 'order_channel', 'last_order_channel', 'first_order_date',
       'last_order_date', 'last_order_date_online', 'last_order_date_offline',
       'order_num_total_ever_online', 'order_num_total_ever_offline',
       'customer_value_total_ever_offline', 'customer_value_total_ever_online',
       'interested_in_categories_12'],
      dtype='str')

In [28]:
df_copy.shape

(19945, 12)

In [29]:
df_copy.describe().T

,count,mean,std,min,25%,50%,75%,max
order_num_total_ever_online,19945.000,3.111,4.226,1.000,1.000,2.000,4.000,200.000
order_num_total_ever_offline,19945.000,1.914,2.063,1.000,1.000,1.000,2.000,109.000
customer_value_total_ever_offline,19945.000,253.923,301.533,10.000,99.990,179.980,319.970,18119.140
customer_value_total_ever_online,19945.000,497.322,832.602,12.990,149.980,286.460,578.440,45220.130


In [30]:
df.isnull().sum()

master_id                            0
order_channel                        0
last_order_channel                   0
first_order_date                     0
last_order_date                      0
last_order_date_online               0
last_order_date_offline              0
order_num_total_ever_online          0
order_num_total_ever_offline         0
customer_value_total_ever_offline    0
customer_value_total_ever_online     0
interested_in_categories_12          0
dtype: int64

In [31]:
df_copy.info()

<class 'pandas.DataFrame'>
RangeIndex: 19945 entries, 0 to 19944
Data columns (total 12 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   master_id                          19945 non-null  str    
 1   order_channel                      19945 non-null  str    
 2   last_order_channel                 19945 non-null  str    
 3   first_order_date                   19945 non-null  str    
 4   last_order_date                    19945 non-null  str    
 5   last_order_date_online             19945 non-null  str    
 6   last_order_date_offline            19945 non-null  str    
 7   order_num_total_ever_online        19945 non-null  float64
 8   order_num_total_ever_offline       19945 non-null  float64
 9   customer_value_total_ever_offline  19945 non-null  float64
 10  customer_value_total_ever_online   19945 non-null  float64
 11  interested_in_categories_12        19945 non-null  str    
dtypes

### 3. Omnichannel Toplam Değişkenleri

Omnichannel müşterilerin hem online'dan hem de offline platformlardan alışveriş yaptığını ifade etmektedir. Her bir müşterinin toplam alışveriş sayısı ve harcaması için yeni değişkenler oluşturunuz.

In [32]:
# Şimdi, her bir müşterinin toplamda kaç sipariş verdiğini ve toplamda ne kadar harcama yaptığını gösterecek iki yeni değişken ekleyeceğiz.
# Online ve offline alışveriş sayıları toplayarak toplam sipariş sayısı 'order_num_total_ever' olarak bir sütun ekliyoruz.
df_copy["order_num_total_ever"] = df_copy["order_num_total_ever_online"] + df_copy["order_num_total_ever_offline"]

# Aynı şekilde, online ve offline harcamaları toplayarak müşterinin toplam harcamasını 'customer_value_total_ever' isimli yeni bir sütunda gösteriyoruz.
df_copy["customer_value_total_ever"] = df_copy["customer_value_total_ever_online"] + df_copy["customer_value_total_ever_offline"]

# Sonuç olarak, ilk birkaç satıra göz atarak yeni sütunlarımızın eklendiğini doğrulayabiliriz.
df_copy.head()

,master_id,order_channel,last_order_channel,first_order_date,last_order_date,last_order_date_online,last_order_date_offline,order_num_total_ever_online,order_num_total_ever_offline,customer_value_total_ever_offline,customer_value_total_ever_online,interested_in_categories_12,order_num_total_ever,customer_value_total_ever
0,cc294636-19f0-11eb-8d74-000d3a38a36f,Android App,Offline,2020-10-30,2021-02-26,2021-02-21,2021-02-26,4.000,1.000,139.990,799.380,[KADIN],5.000,939.370
1,f431bd5a-ab7b-11e9-a2fc-000d3a38a36f,Android App,Mobile,2017-02-08,2021-02-16,2021-02-16,2020-01-10,19.000,2.000,159.970,1853.580,"[ERKEK, COCUK, KADIN, AKTIFSPOR]",21.000,2013.550
2,69b69676-1a40-11ea-941b-000d3a38a36f,Android App,Android App,2019-11-27,2020-11-27,2020-11-27,2019-12-01,3.000,2.000,189.970,395.350,"[ERKEK, KADIN]",5.000,585.320
3,1854e56c-491f-11eb-806e-000d3a38a36f,Android App,Android App,2021-01-06,2021-01-17,2021-01-17,2021-01-06,1.000,1.000,39.990,81.980,"[AKTIFCOCUK, COCUK]",2.000,121.970
4,d6ea1074-f1f5-11e9-9346-000d3a38a36f,Desktop,Desktop,2019-08-03,2021-03-07,2021-03-07,2019-08-03,1.000,1.000,49.990,159.990,[AKTIFSPOR],2.000,209.980


### 4. Tarih Değişkenlerinin Dönüştürülmesi

Değişken tiplerini inceleyiniz. Tarih ifade eden değişkenlerin tipini `date`'e çeviriniz.

In [33]:
# Tarih ifade eden değişkenleri date tipine çeviriniz.
date_cols = ["first_order_date", "last_order_date", "last_order_date_online", "last_order_date_offline"]
for col in date_cols:
    df_copy[col] = pd.to_datetime(df_copy[col])

df_copy.info()

<class 'pandas.DataFrame'>
RangeIndex: 19945 entries, 0 to 19944
Data columns (total 14 columns):
 #   Column                             Non-Null Count  Dtype         
---  ------                             --------------  -----         
 0   master_id                          19945 non-null  str           
 1   order_channel                      19945 non-null  str           
 2   last_order_channel                 19945 non-null  str           
 3   first_order_date                   19945 non-null  datetime64[us]
 4   last_order_date                    19945 non-null  datetime64[us]
 5   last_order_date_online             19945 non-null  datetime64[us]
 6   last_order_date_offline            19945 non-null  datetime64[us]
 7   order_num_total_ever_online        19945 non-null  float64       
 8   order_num_total_ever_offline       19945 non-null  float64       
 9   customer_value_total_ever_offline  19945 non-null  float64       
 10  customer_value_total_ever_online   19945 non-

### 5. Alışveriş Kanalları Dağılımı

Alışveriş kanallarındaki müşteri sayısının, toplam alınan ürün sayısı ve toplam harcamaların dağılımına bakınız.

In [34]:
# Alışveriş kanallarındaki dağılımı inceleyelim:
# - Her alışveriş kanalında (order_channel) toplam kaç eşsiz müşteri bulunuyor?
# - Tüm müşterilerin o kanaldan verdiği toplam sipariş (order_num_total_ever) sayısı ne?
# - O kanalda, müşterilerin yaptığı toplam harcama (customer_value_total_ever) tutarı ne?
# Bunu iyi görmek için groupby ile özet tablo (aggregation) oluşturuyoruz.

channel_agg = df_copy.groupby("order_channel").agg(
    customer_count=("master_id", "nunique"),  # Her kanalda benzersiz müşteri sayısı
    total_order_count=("order_num_total_ever", "sum"),  # Toplam sipariş sayısı
    total_value=("customer_value_total_ever", "sum"),   # Toplam harcama
)

# Son tabloyu yazdırıyoruz:
print("Alışveriş Kanallarına Göre Özet Tablo:")
print(channel_agg)

Alışveriş Kanallarına Göre Özet Tablo:
               customer_count  total_order_count  total_value
order_channel                                                
Android App              9495          52269.000  7819062.760
Desktop                  2735          10920.000  1610321.460
Ios App                  2833          15351.000  2525999.930
Mobile                   4882          21679.000  3028183.160


### 6. En Fazla Kazanç Getiren İlk 10 Müşteri

En fazla kazancı getiren ilk 10 müşteriyi sıralayınız.

In [35]:
# 'customer_value_total_ever' sütununa göre azalan şekilde sıralayıp ilk 10 satırı seçiyoruz.
# Bunun için veri çerçevemizi ('df_copy'), 'customer_value_total_ever' değerlerine göre büyükten küçüğe sıralıyoruz.
# Daha sonra 'head(10)' fonksiyonu ile ilk 10 müşteriyi seçiyoruz.
# Sonuç olarak, en fazla para harcayan 10 müşterinin 'master_id' ve 'customer_value_total_ever' sütunlarını ekrana yazdırıyoruz.

top10_value_customers = df_copy.sort_values("customer_value_total_ever", ascending=False).head(10)
print(top10_value_customers[["master_id", "customer_value_total_ever"]])

                                  master_id  customer_value_total_ever
11150  5d1c466a-9cfd-11e9-9897-000d3a38a36f                  45905.100
4315   d5ef8058-a5c6-11e9-a2fc-000d3a38a36f                  36818.290
7613   73fd19aa-9e37-11e9-9897-000d3a38a36f                  33918.100
13880  7137a5c0-7aad-11ea-8f20-000d3a38a36f                  31227.410
9055   47a642fe-975b-11eb-8c2a-000d3a38a36f                  20706.340
7330   a4d534a2-5b1b-11eb-8dbd-000d3a38a36f                  18443.570
8068   d696c654-2633-11ea-8e1c-000d3a38a36f                  16918.570
163    fef57ffa-aae6-11e9-a2fc-000d3a38a36f                  12726.100
7223   cba59206-9dd1-11e9-9897-000d3a38a36f                  12282.240
18767  fc0ce7a4-9d87-11e9-9897-000d3a38a36f                  12103.150


### 7. En Fazla Sipariş Veren İlk 10 Müşteri

En fazla siparişi veren ilk 10 müşteriyi sıralayınız.

In [36]:
# En fazla siparişi veren ilk 10 müşteriyi bulmak için, 'order_num_total_ever' değişkenini kullanıyoruz.
# 'order_num_total_ever' sütununa göre azalan şekilde sıralayıp, ilk 10 müşteriyi seçiyoruz.
# Sonrasında, bu müşterilerin 'master_id' ve 'order_num_total_ever' bilgilerine bakıyoruz.

top10_order_customers = df_copy.sort_values("order_num_total_ever", ascending=False).head(10)
top10_order_customers[["master_id", "order_num_total_ever"]]

,master_id,order_num_total_ever
11150,5d1c466a-9cfd-11e9-9897-000d3a38a36f,202.000
7223,cba59206-9dd1-11e9-9897-000d3a38a36f,131.000
8783,a57f4302-b1a8-11e9-89fa-000d3a38a36f,111.000
2619,fdbe8304-a7ab-11e9-a2fc-000d3a38a36f,88.000
6322,329968c6-a0e2-11e9-a2fc-000d3a38a36f,83.000
7613,73fd19aa-9e37-11e9-9897-000d3a38a36f,82.000
9347,44d032ee-a0d4-11e9-a2fc-000d3a38a36f,77.000
10954,b27e241a-a901-11e9-a2fc-000d3a38a36f,75.000
8068,d696c654-2633-11ea-8e1c-000d3a38a36f,70.000
7330,a4d534a2-5b1b-11eb-8dbd-000d3a38a36f,70.000


### 8. Veri Ön Hazırlık Fonksiyonu

Veri ön hazırlık sürecini fonksiyonlaştırınız.

In [ ]:
def data_preprocessing(df):
    """
    FLO müşteri veri seti için veri ön hazırlık işlemlerini gerçekleştirir.
    1. Tarih değişkenlerini tarih (datetime) tipine çevirir.
    2. Toplam sipariş ve toplam harcama için yeni değişkenler oluşturur.
    
    Parametreler:
        df (pd.DataFrame): Analiz edilecek veri çerçevesi.
        
    Dönüş:
        pd.DataFrame: Ön işlenmiş ve yeni değişkenleri eklenmiş veri çerçevesi.
    """
    import pandas as pd  # Notbook üstünde import edildi ise buraya yazmanıza gerek yok.

    # Tarih olan değişkenleri datetime'a çevir.
    date_cols = [col for col in df.columns if "date" in col]
    for col in date_cols:
        df[col] = pd.to_datetime(df[col])
    
    # Online ve offline toplam alışveriş adetini veren yeni değişken
    df["order_num_total_ever"] = df["order_num_total_ever_online"] + df["order_num_total_ever_offline"]
    
    # Online ve offline toplam harcama tutarını veren yeni değişken
    df["customer_value_total_ever"] = df["customer_value_total_ever_online"] + df["customer_value_total_ever_offline"]
    
    return df

data_preprocessing(df_copy)

,master_id,order_channel,last_order_channel,first_order_date,last_order_date,last_order_date_online,last_order_date_offline,order_num_total_ever_online,order_num_total_ever_offline,customer_value_total_ever_offline,customer_value_total_ever_online,interested_in_categories_12,order_num_total_ever,customer_value_total_ever
0,cc294636-19f0-11eb-8d74-000d3a38a36f,Android App,Offline,2020-10-30,2021-02-26,2021-02-21,2021-02-26,4.000,1.000,139.990,799.380,[KADIN],5.000,939.370
1,f431bd5a-ab7b-11e9-a2fc-000d3a38a36f,Android App,Mobile,2017-02-08,2021-02-16,2021-02-16,2020-01-10,19.000,2.000,159.970,1853.580,"[ERKEK, COCUK, KADIN, AKTIFSPOR]",21.000,2013.550
2,69b69676-1a40-11ea-941b-000d3a38a36f,Android App,Android App,2019-11-27,2020-11-27,2020-11-27,2019-12-01,3.000,2.000,189.970,395.350,"[ERKEK, KADIN]",5.000,585.320
3,1854e56c-491f-11eb-806e-000d3a38a36f,Android App,Android App,2021-01-06,2021-01-17,2021-01-17,2021-01-06,1.000,1.000,39.990,81.980,"[AKTIFCOCUK, COCUK]",2.000,121.970
4,d6ea1074-f1f5-11e9-9346-000d3a38a36f,Desktop,Desktop,2019-08-03,2021-03-07,2021-03-07,2019-08-03,1.000,1.000,49.990,159.990,[AKTIFSPOR],2.000,209.980
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19940,727e2b6e-ddd4-11e9-a848-000d3a38a36f,Android App,Offline,2019-09-21,2020-07-05,2020-06-05,2020-07-05,1.000,2.000,289.980,111.980,"[ERKEK, AKTIFSPOR]",3.000,401.960
19941,25cd53d4-61bf-11ea-8dd8-000d3a38a36f,Desktop,Desktop,2020-03-01,2020-12-22,2020-12-22,2020-03-01,1.000,1.000,150.480,239.990,[AKTIFSPOR],2.000,390.470
19942,8aea4c2a-d6fc-11e9-93bc-000d3a38a36f,Ios App,Ios App,2019-09-11,2021-05-24,2021-05-24,2019-09-11,2.000,1.000,139.980,492.960,[AKTIFSPOR],3.000,632.940
19943,e50bb46c-ff30-11e9-a5e8-000d3a38a36f,Android App,Android App,2019-03-27,2021-02-13,2021-02-13,2021-01-08,1.000,5.000,711.790,297.980,"[ERKEK, AKTIFSPOR]",6.000,1009.770


---
## GÖREV 2: RFM Metriklerinin Hesaplanması

Adım 1: Recency, Frequency ve Monetary tanımlarını yapınız.

Adım 2: Müşteri özelinde Recency, Frequency ve Monetary metriklerini hesaplayınız.

Adım 3: Hesapladığınız metrikleri rfm isimli bir değişkene atayınız.

Adım 4: Oluşturduğunuz metriklerin isimlerini recency, frequency ve monetary olarak değiştiriniz. 

recency değerini hesaplamak için analiz tarihini maksimum tarihten 2 gün sonrası seçebilirsiniz

In [ ]:
# Adım 1: Recency, Frequency ve Monetary tanımlarını yapalım.
# - Recency: Müşterinin en son alışverişinden bu yana geçen gün sayısıdır (analiz tarihi - müşterinin son alışveriş tarihi).
# - Frequency: Müşterinin toplam satın alma (alışveriş) sayısıdır.
# - Monetary: Müşterinin toplam harcama (para) tutarıdır.

# Adım 2: Analiz tarihi olarak, veri setindeki son sipariş tarihinden 2 gün sonrasını alalım.
analysis_date = df_copy["last_order_date"].max() + pd.DateOffset(days=2)

# Adım 3: Müşteri özelinde Recency, Frequency ve Monetary metriklerini hesaplayalım.
rfm = df_copy.groupby("master_id").agg(
    recency = ("last_order_date", lambda x: (analysis_date - x.max()).days),
    frequency = ("order_num_total_ever", "sum"),
    monetary = ("customer_value_total_ever", "sum")
).reset_index()

# Adım 4: Kolon isimlerini düzenleyelim (zaten uygun şekilde oluşturuldu).
rfm.rename(columns={"master_id": "customer_id"}, inplace=True)

# Sonucu görüntüleyelim.
rfm.head()


,customer_id,recency,frequency,monetary
0,00016786-2f5a-11ea-bb80-000d3a38a36f,10,5.000,776.070
1,00034aaa-a838-11e9-a2fc-000d3a38a36f,298,3.000,269.470
2,000be838-85df-11ea-a90b-000d3a38a36f,213,4.000,722.690
3,000c1fe2-a8b7-11ea-8479-000d3a38a36f,27,7.000,874.160
4,000f5e3e-9dde-11ea-80cd-000d3a38a36f,20,7.000,1620.330


---
## GÖREV 3: RF ve RFM Skorlarının Hesaplanması (Calculating RF and RFM Scores)

- Recency, Frequency ve Monetary metriklerini `qcut` yardımı ile 1-5 arasında skorlara çeviriniz.
- Bu skorları `recency_score`, `frequency_score` ve `monetary_score` olarak kaydediniz.
- `recency_score` ve `frequency_score`'u tek bir değişken olarak ifade ediniz ve `RF_SCORE` olarak kaydediniz.

In [39]:
# Adım 1: Recency, Frequency ve Monetary metriklerini 1-5 arası değerlere dönüştürmek için qcut fonksiyonunu kullanıyoruz.
# Recency'de, değeri düşük olanlar yani daha yakın zamanda işlem yapanlar daha yüksek skor almalıdır. 
# Bu yüzden recency'de en düşük değerlere 5, en yüksek değerlere 1 puan veriyoruz (labels=[5,4,3,2,1]).
rfm['recency_score'] = pd.qcut(rfm['recency'], 5, labels=[5,4,3,2,1]).astype(int)

# Frequency için, çok alışveriş yapanlar yüksek skor almalı.
# Aynı skor oluşmasını engellemek için rank() fonksiyonu ile önce sıralama yapıp ardından qcut ile dilimleyerek 1 en düşük, 5 en yüksek olacak şekilde etiketliyoruz.
rfm['frequency_score'] = pd.qcut(rfm['frequency'].rank(method='first'), 5, labels=[1,2,3,4,5]).astype(int)

# Monetary'de ise fazla harcayanlara yüksek skor vermek için aynı şekilde qcut ile 5 parçaya bölüp, fazla harcayanlara 5, az harcayanlara 1 puan atıyoruz.
rfm['monetary_score'] = pd.qcut(rfm['monetary'], 5, labels=[1,2,3,4,5]).astype(int)

# Adım 2: RF_SCORE oluşturulması.
# recency_score ve frequency_score'u string olarak birleştirip, her müşteri için iki haneli bir skor elde ediyoruz (ör: 55, 43 gibi).
rfm['RF_SCORE'] = rfm['recency_score'].astype(str) + rfm['frequency_score'].astype(str)

# Sonuç kontrolü için ilk birkaç satırı görüntülüyoruz.
rfm.head()

,customer_id,recency,frequency,monetary,recency_score,frequency_score,monetary_score,RF_SCORE
0,00016786-2f5a-11ea-bb80-000d3a38a36f,10,5.000,776.070,5,4,4,54
1,00034aaa-a838-11e9-a2fc-000d3a38a36f,298,3.000,269.470,1,2,1,12
2,000be838-85df-11ea-a90b-000d3a38a36f,213,4.000,722.690,2,3,4,23
3,000c1fe2-a8b7-11ea-8479-000d3a38a36f,27,7.000,874.160,5,4,4,54
4,000f5e3e-9dde-11ea-80cd-000d3a38a36f,20,7.000,1620.330,5,4,5,54


---
## GÖREV 4: RF Skorlarının Segment Olarak Tanımlanması

Oluşturulan RFM skorların daha açıklanabilir olması için segment tanımlayınız ve tanımlanan `seg_map` yardımı ile `RF_SCORE`'u segmentlere çeviriniz.

In [ ]:
# Segmentasyon adımı burada gerçekleştirilir.
# Her müşterinin RF_SCORE (recency ve frequency skorunun birleşimi), segment isimleriyle eşleştirilerek müşteri davranış profili belirlenir.
# Aşağıdaki seg_map sözlüğünde, regex şablonları her segmentin hangi RF_SCORE aralıklarına karşılık geldiğini belirtir.
# Örneğin, '[1-2][1-2]' deseni hem recency hem frequency düşük olan (yani uzun süre alışveriş yapmamış ve seyrek alışveriş yapan) 'hibernating' (uyuyan) müşteri segmentini temsil eder.

seg_map = {
    r'[1-2][1-2]': 'hibernating',            # Hem recency hem frequency skorları en düşük olan müşteriler
    r'[1-2][3-4]': 'at_RISK',                # Uzun süredir alışveriş yapmamış ama geçmişte nispeten iyi müşteri olanlar
    r'[1-2]5': 'cant_lose',                  # Çok uzun süredir alışveriş yapmamış ancak çok sık alışveriş yapmış; kaybedilmemesi gereken müşteriler
    r'3[1-2]': 'about_to_sleep',             # Bir süredir işlem yapmayan, uyumaya yatmak üzere olan müşteriler
    r'33': 'need_attention',                 # Dikkat edilmesi gereken (ilgiye muhtaç) müşteriler
    r'[3-4][4-5]': 'loyal_customers',        # Sık alışveriş yapan ve orta derecede yeni olan sadık müşteriler
    r'[4-5][1-2]': 'promising',              # Yenilere yakın ancak alışveriş sıklığı düşük müşteriler
    r'51': 'new_customers',                  # En son alışveriş yapan yeni müşteriler ancak henüz sık alışveriş yapmıyorlar
    r'[4-5][3]': 'potential_loyalists',      # Yüksek recency ve orta frequency — potansiyel sadık müşteri adayı
    r'5[4-5]': 'champions'                   # Hem çok yakın zamanda hem de çok sık alışveriş yapan en değerli müşteriler
}

# Şimdi RF_SCORE değerini yukarıdaki segment isimlerine çevirecek bir fonksiyon tanımlanıyor.
import re

def rf_segment(row):
    # RF_SCORE'u segment pattern'larına göre kontrol et ve uygun segmente ata
    for pattern, segment in seg_map.items():
        if re.match(pattern, row):
            return segment
    return 'other'  # RF_SCORE hiçbir pattern ile eşleşmezse 'other' segmentine atanır

# Her müşteri için RF_SCORE bazında segment bilgisini dataframe'e ekle
rfm['segment'] = rfm['RF_SCORE'].apply(rf_segment)

# Son olarak, segmentlerin dağılımını value_counts() ile analiz ediyoruz
rfm['segment'].value_counts()


segment
hibernating            3589
loyal_customers        3375
at_RISK                3152
promising              2746
champions              1920
about_to_sleep         1643
potential_loyalists    1520
cant_lose              1194
need_attention          806
Name: count, dtype: int64

---
## GÖREV 5: Aksiyon Zamanı!

### 1. Segment Ortalamaları

Segmentlerin recency, frequency ve monetary ortalamalarını inceleyiniz.

In [41]:
# Her bir müşteri segmentinin recency (müşterinin en son alışverişinden bu yana geçen süre), frequency (alışveriş sıklığı) ve monetary (toplam harcama) ortalamalarını analiz etmek için
# rfm tablosunda 'segment' değişkenine göre gruplama yapılıyor.
# Ardından bu grupların ilgili ortalama değerleri hesaplanıyor ve daha okunaklı olması için iki ondalık basamağa yuvarlanıyor.
segment_ortalamalari = rfm.groupby('segment')[['recency', 'frequency', 'monetary']].mean().round(2)
segment_ortalamalari

,recency,frequency,monetary
segment,,,
about_to_sleep,114.030,2.410,361.650
at_RISK,242.330,4.470,648.330
cant_lose,235.160,10.720,1481.650
champions,17.140,8.970,1410.710
hibernating,247.430,2.390,362.580
loyal_customers,82.560,8.360,1216.260
need_attention,113.040,3.740,553.440
potential_loyalists,36.840,3.740,604.020
promising,37.570,2.430,399.800


### 2a. Yeni Kadın Ayakkabı Markası Hedef Müşterileri

FLO bünyesine yeni bir kadın ayakkabı markası dahil ediyor. Dahil ettiği markanın ürün fiyatları genel müşteri tercihlerinin üstünde. Bu nedenle markanın tanıtımı ve ürün satışları için ilgilenecek profildeki müşterilerle özel olarak iletişime geçilmek isteniliyor.

**Hedef profil:** Sadık müşteriler (champions, loyal_customers), ortalama 250 TL üzeri ve kadın kategorisinden alışveriş yapan kişiler.

Bu müşterilerin id numaralarını `yeni_marka_hedef_müşteri_id.csv` dosyasına kaydediniz.

In [ ]:
# Sadık müşteriler (champions, loyal_customers) ve ortalama 250 TL üzeri, kadın kategorisinden alışveriş yapanlar bulunacak

# 1. Sadık müşteri segmentleri seç
hedef_segmentler = ["champions", "loyal_customers"]
hedef_musteriler = rfm[rfm["segment"].isin(hedef_segmentler)].copy()

# 2. Ortalama harcaması 250 TL üzeri olan müşterileri filtrele
hedef_musteriler = hedef_musteriler[hedef_musteriler["monetary"] > 250]

# 3. main_data'da kadın kategorisinden alışveriş yapan müşteri id'lerini seç
# 'main_data' tabloda "interested_in_categories_12" kolonu olduğunu varsayıyoruz ve bu kolon string/list türünde
kadin_kategorisi_olan_idler = df_copy[df_copy["interested_in_categories_12"].apply(lambda x: "KADIN" in str(x))]["master_id"].unique()

# 4. Hem hedef segment hem de kadın kategorisinden alışveriş yapanları bul
yeni_marka_hedef_ids = hedef_musteriler[hedef_musteriler["customer_id"].isin(kadin_kategorisi_olan_idler)]["customer_id"]

# 5. Sonucu CSV olarak kaydet
yeni_marka_hedef_ids.to_frame(name="customer_id").to_csv("yeni_marka_hedef_musteri_id.csv", index=False)

### 2b. Erkek ve Çocuk İndirim Hedef Müşterileri

Erkek ve Çocuk ürünlerinde %40'a yakın indirim planlanmaktadır. Bu indirimle ilgili kategorilerle ilgilenen; geçmişte iyi müşteri olan ama uzun süredir alışveriş yapmayan, kaybedilmemesi gereken müşteriler, uykuda olanlar ve yeni gelen müşteriler özel olarak hedef alınmak isteniliyor.

Uygun profildeki müşterilerin id'lerini `indirim_hedef_müşteri_ids.csv` dosyasına kaydediniz.

In [ ]:
# Erkek ve Çocuk indirim hedef müşteri profili:
# Kriterler:
# - 'interested_in_categories_12' sütununda "ERKEK" veya "COCUK" olanlar
# - RFM segmenti: 'hibernating', 'about_to_sleep', 'cant_lose', 'at_RISK', 'new_customers' segmentlerinde olanlar

indirim_hedef_segmentler = ["hibernating", "about_to_sleep", "cant_lose", "at_RISK", "new_customers"]

# Ana veri tablosunda ilgilendiği kategorilere 'ERKEK' veya 'COCUK' olan müşterileri seç
erkek_veya_cocuk_kategorisi_idler = df_copy[
    df_copy["interested_in_categories_12"].apply(lambda x: ("ERKEK" in str(x)) or ("COCUK" in str(x)))
]["master_id"].unique()

# Hedef RFM segmentlerindeki müşteriler
indirim_hedef_musteriler = rfm[rfm["segment"].isin(indirim_hedef_segmentler)]

# Her iki kritere uyanları bul
indirim_hedef_ids = indirim_hedef_musteriler[
    indirim_hedef_musteriler["customer_id"].isin(erkek_veya_cocuk_kategorisi_idler)
]["customer_id"]

# Sonucu CSV olarak kaydet
indirim_hedef_ids.to_frame(name="customer_id").to_csv("indirim_hedef_musteri_ids.csv", index=False)

---
## GÖREV 6: Tüm Süreci Fonksiyonlaştırma

Yukarıdaki tüm analiz sürecini tek bir fonksiyon veya pipeline halinde fonksiyonlaştırınız.

In [48]:
def flo_rfm_customer_segmentation_pipeline(
    csv_path="datasets/flo_data_20k.csv",
    yeni_marka_hedef_csv="yeni_marka_hedef_musteri_id.csv",
    indirim_hedef_csv="indirim_hedef_musteri_ids.csv"
):
    """
    Flo RFM Müşteri Segmentasyonu sürecini baştan sona uygulayan fonksiyondur.

    Parametreler:
        csv_path (str): Müşteri ve satış verisinin bulunduğu CSV dosyasının yolu.
        yeni_marka_hedef_csv (str): KADIN yeni marka hedef müşteri ID'lerinin yazılacağı CSV dosyası.
        indirim_hedef_csv (str): Erkek & Çocuk indirim hedef müşteri ID'lerinin yazılacağı CSV dosyası.

    Fonksiyonun Yaptıkları:
        1. CSV dosyasından veriyi okur.
        2. Müşteri lifetime metriklerini hesaplar: toplam alışveriş adedi ve toplam harcama tutarı.
        3. Tarih ile ilgili sütunları pandas datetime tipine çevirir.
        4. RFM tablosu kurar: recency (son alışverişten bugüne geçen gün), frequency (toplam alışveriş sayısı), monetary (toplam harcama).
        5. Recency, frequency ve monetary skorlarını 1-5 arasında bölerek atar.
        6. RF_SCORE ile regex pattern'lara göre müşteri segmentelerini (ör: loyal_customers, champions, hibernating vs.) belirler.
        7a. Kadın ürün grubunda,
            - Segmenti 'champions', 'loyal_customers', 'potential_loyalists' olan
            - Ve 'KADIN' kategorisiyle ilgilenen müşterilerin ID'lerini csv'ye kaydeder.
        7b. Erkek/Çocuk ürünlerinde indirim hedefi için,
            - Segmenti "hibernating", "about_to_sleep", "cant_lose", "at_RISK", "new_customers" olup
            - 'interested_in_categories_12' de "ERKEK" veya "COCUK" geçen müşteri ID'lerini csv'ye kaydeder.
        8. RFM tablosu ve iki hedef müşteri grubunu döndürür.

    Returns:
        dict: {"rfm": rfm, "yeni_marka_hedef_ids": ..., "indirim_hedef_ids": ...}
    """

    import pandas as pd
    import datetime as dt

    # 1. Veriyi oku ve ön hazırlık
    df = pd.read_csv(csv_path)
    df.columns = [col.lower() for col in df.columns]  # Sütun isimlerini küçült

    # 2. Lifetime metrikleri
    df["order_num_total_ever_online"] = df["order_num_total_ever_online"].astype(float)
    df["order_num_total_ever_offline"] = df["order_num_total_ever_offline"].astype(float)
    df["customer_value_total_ever_offline"] = df["customer_value_total_ever_offline"].astype(float)
    df["customer_value_total_ever_online"] = df["customer_value_total_ever_online"].astype(float)
    df["order_num_total_ever"] = df["order_num_total_ever_online"] + df["order_num_total_ever_offline"]
    df["customer_value_total_ever"] = df["customer_value_total_ever_offline"] + df["customer_value_total_ever_online"]

    # 3. Tarih sütunlarını datetime formatına çevir
    date_cols = [col for col in df.columns if "date" in col]
    for col in date_cols:
        df[col] = pd.to_datetime(df[col])

    # 4. RFM tablosunu oluştur
    today_date = dt.datetime(2021, 6, 1)  # Sabit analiz tarihi
    rfm = pd.DataFrame()
    rfm["customer_id"] = df["master_id"]
    rfm["recency"] = (today_date - df["last_order_date"]).dt.days
    rfm["frequency"] = df["order_num_total_ever"]
    rfm["monetary"] = df["customer_value_total_ever"]

    # 5. RFM skorları: Recency düşükse (yeniyse) skor 5, gün yüksekse skor 1 olur.
    rfm["recency_score"] = pd.qcut(rfm["recency"], 5, labels=[5, 4, 3, 2, 1]).astype(int)
    # Frequency ile monetary, yüksekse skor büyük (5), düşükse küçük (1)
    rfm["frequency_score"] = pd.qcut(rfm["frequency"].rank(method="first"), 5, labels=[1,2,3,4,5]).astype(int)
    rfm["monetary_score"] = pd.qcut(rfm["monetary"], 5, labels=[1,2,3,4,5]).astype(int)

    rfm["RF_SCORE"] = rfm["recency_score"].astype(str) + rfm["frequency_score"].astype(str)

    # 6. Segment tanımları: RF_SCORE'a göre regex ile segment ataması
    seg_map = {
        r'[1-2][1-2]': 'hibernating',          # En sadık olmayanlar, alışverişten çok zaman geçmiş, az alışveriş
        r'[1-2][3-4]': 'at_RISK',              # Riskli, kaybediliyor
        r'[1-2]5': 'cant_lose',                # Kaybedilmemesi gerekenler
        r'3[1-2]': 'about_to_sleep',           # Uykuda olan
        r'33': 'need_attention',               # İlgilenilmesi gerekenler
        r'[3-4][4-5]': 'loyal_customers',      # Sadık
        r'41': 'promising',                    # Potansiyeli var, yeni gelebilir
        r'51': 'promising',
        r'[4-5][2-3]': 'potential_loyalists',  # Sadık müşteri olma potansiyeli
        r'5[4-5]': 'champions'                 # En iyi, en yeni ve en çok alışveriş yapanlar
    }

    def segment(row):
        """Her müşteri için regex ile segment adını döndür. Hiçbiri uymuyorsa new_customers."""
        for pattern, seg in seg_map.items():
            if pd.Series(row["RF_SCORE"]).str.contains(pattern, regex=True)[0]:
                return seg
        return "new_customers"

    rfm["segment"] = rfm.apply(segment, axis=1)

    # 7a. Yeni marka hedefi (KADIN ilgisi ve üst segmentten olanlar)
    yeni_marka_segmentler = ["champions", "loyal_customers", "potential_loyalists"]
    kadin_kategorisi_idler = df[
        df["interested_in_categories_12"].apply(lambda x: "KADIN" in str(x))
    ]["master_id"].unique()
    yeni_marka_hedef_ids = rfm[
        (rfm["segment"].isin(yeni_marka_segmentler)) &
        (rfm["customer_id"].isin(kadin_kategorisi_idler))
    ]["customer_id"]
    yeni_marka_hedef_ids.to_frame(name="customer_id").to_csv(yeni_marka_hedef_csv, index=False)

    # 7b. Erkek & Çocuk ürünlerinde indirim hedefi (ilgili kategori + segment)
    indirim_hedef_segmentler = ["hibernating", "about_to_sleep", "cant_lose", "at_RISK", "new_customers"]
    erkek_veya_cocuk_kategorisi_idler = df[
        df["interested_in_categories_12"].apply(lambda x: ("ERKEK" in str(x)) or ("COCUK" in str(x)))
    ]["master_id"].unique()
    indirim_hedef_musteriler = rfm[rfm["segment"].isin(indirim_hedef_segmentler)]
    indirim_hedef_ids = indirim_hedef_musteriler[
        indirim_hedef_musteriler["customer_id"].isin(erkek_veya_cocuk_kategorisi_idler)
    ]["customer_id"]
    indirim_hedef_ids.to_frame(name="customer_id").to_csv(indirim_hedef_csv, index=False)

    # 8. Sonuç: Tüm RFM tablosu ve segmentasyon sonuç grupları
    return {
        "rfm": rfm,
        "yeni_marka_hedef_ids": yeni_marka_hedef_ids,
        "indirim_hedef_ids": indirim_hedef_ids
    }

flo_rfm_customer_segmentation_pipeline(csv_path="datasets/flo_data_20k.csv", yeni_marka_hedef_csv="yeni_marka_hedef_musteri_id.csv", indirim_hedef_csv="indirim_hedef_musteri_ids.csv")

{'rfm':                                 customer_id  recency  frequency  monetary  \
 0      cc294636-19f0-11eb-8d74-000d3a38a36f       95      5.000   939.370   
 1      f431bd5a-ab7b-11e9-a2fc-000d3a38a36f      105     21.000  2013.550   
 2      69b69676-1a40-11ea-941b-000d3a38a36f      186      5.000   585.320   
 3      1854e56c-491f-11eb-806e-000d3a38a36f      135      2.000   121.970   
 4      d6ea1074-f1f5-11e9-9346-000d3a38a36f       86      2.000   209.980   
 ...                                     ...      ...        ...       ...   
 19940  727e2b6e-ddd4-11e9-a848-000d3a38a36f      331      3.000   401.960   
 19941  25cd53d4-61bf-11ea-8dd8-000d3a38a36f      161      2.000   390.470   
 19942  8aea4c2a-d6fc-11e9-93bc-000d3a38a36f        8      3.000   632.940   
 19943  e50bb46c-ff30-11e9-a5e8-000d3a38a36f      108      6.000  1009.770   
 19944  740998d2-b1f7-11e9-89fa-000d3a38a36f      360      2.000   261.970   
 
        recency_score  frequency_score  monetary_score 